# SIS №1: Data Collection & Preparation
**Economy and Population: how GDP relates to human development**

Tajikossov Nurbolsyn, 24B032042, team: nurbo team

Question: do countries with higher GDP per capita also have a higher HDI and longer life expectancy, and how does population growth fit in?

Data:
- API: World Bank Indicators API (GDP per capita, population, life expectancy, population growth, unemployment)
- Web scraping: Wikipedia table "List of countries by Human Development Index"

Steps: collect, clean, merge, EDA, visualisation.

In [ ]:
import re
import time
import unicodedata
import urllib.robotparser
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
from scipy import stats

START_YEAR, END_YEAR = 2000, 2023
ANALYSIS_YEAR = 2022

WB_BASE = "https://api.worldbank.org/v2"
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_countries_by_Human_Development_Index"

INDICATORS = {
    "NY.GDP.PCAP.CD": "gdp_per_capita",
    "SP.POP.TOTL": "population",
    "SP.DYN.LE00.IN": "life_expectancy",
    "SP.POP.GROW": "pop_growth",
    "SL.UEM.TOTL.ZS": "unemployment",
}

HEADERS = {"User-Agent": "Mozilla/5.0 (student project, KBTU SIS-1)"}

DATA_DIR, FIG_DIR = Path("data"), Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 30)

session = requests.Session()
session.headers.update(HEADERS)

## 1. API: World Bank

The API returns a list of two elements: pagination info and the data itself. It also returns aggregates (World, Euro area, etc.), so country metadata is downloaded separately to filter them out.

In [ ]:
def get_json(url, params=None, retries=3):
    for attempt in range(retries):
        try:
            response = session.get(url, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as e:
            print("request failed:", e)
            time.sleep(2)
    raise RuntimeError("could not fetch " + url)


def fetch_countries():
    items = get_json(f"{WB_BASE}/country", {"format": "json", "per_page": 400})[1]
    meta = pd.DataFrame(
        {"iso3": c["id"], "country": c["name"],
         "region": c["region"]["value"], "income_group": c["incomeLevel"]["value"]}
        for c in items
    )
    return meta[meta["region"] != "Aggregates"].reset_index(drop=True)


def fetch_indicator(code, column):
    rows, page = [], 1
    while True:
        payload = get_json(
            f"{WB_BASE}/country/all/indicator/{code}",
            {"format": "json", "per_page": 20000, "date": f"{START_YEAR}:{END_YEAR}", "page": page},
        )
        if len(payload) < 2 or payload[1] is None:
            raise RuntimeError(f"bad API answer for {code}: {payload[0]}")
        meta, data = payload
        rows += [{"iso3": d["countryiso3code"], "year": d["date"], column: d["value"]} for d in data]
        if page >= meta["pages"]:
            break
        page += 1
    return pd.DataFrame(rows)


countries_meta = fetch_countries()
print("countries:", len(countries_meta))

frames = {}
for code, column in INDICATORS.items():
    frames[column] = fetch_indicator(code, column)
    print(code, "->", column, len(frames[column]), "rows")

In [ ]:
# one table: one row = one country in one year
panel_raw = None
for column, frame in frames.items():
    panel_raw = frame if panel_raw is None else panel_raw.merge(frame, on=["iso3", "year"], how="outer")

print(panel_raw.shape)
display(panel_raw.head())
print(panel_raw.dtypes)   # year is a string here

## 2. Web scraping: Wikipedia HDI table

First I check robots.txt. The parser does not rely on column numbers: for every table row it takes the first cell with letters as the country and the first cell that looks like `0.xxx` as the HDI.

In [ ]:
def allowed_by_robots(url):
    root = re.match(r"https?://[^/]+", url).group(0)
    text = session.get(f"{root}/robots.txt", timeout=30).text
    parser = urllib.robotparser.RobotFileParser()
    parser.parse(text.splitlines())
    return parser.can_fetch(HEADERS["User-Agent"], url)


def clean_cell(text):
    text = re.sub(r"\[.*?\]", "", text).replace("\xa0", " ")   # remove footnotes like [1]
    return re.sub(r"\s+", " ", text).strip()


def scrape_hdi(url):
    page = session.get(url, timeout=30)
    page.raise_for_status()
    soup = BeautifulSoup(page.text, "html.parser")

    records = []
    for table in soup.select("table.wikitable"):
        for tr in table.find_all("tr"):
            texts = [clean_cell(c.get_text(" ", strip=True)) for c in tr.find_all(["th", "td"])]
            hdi = next((float(t) for t in texts if re.fullmatch(r"0\.\d{3}", t)), None)
            name = next((t for t in texts if re.search(r"[A-Za-z]{3}", t)), None)
            if hdi is not None and name is not None:
                records.append({"wiki_country": name, "hdi": hdi})
    return pd.DataFrame(records)


print("allowed by robots.txt:", allowed_by_robots(WIKI_URL))
wiki_raw = scrape_hdi(WIKI_URL)
print("scraped rows:", len(wiki_raw))
display(wiki_raw.head())
assert len(wiki_raw) > 150, "too few rows, check the page layout"

## 3. Cleaning
### 3.1 World Bank data

In [ ]:
panel = panel_raw.copy()

# data types
panel["year"] = pd.to_numeric(panel["year"], errors="coerce")
for column in INDICATORS.values():
    panel[column] = pd.to_numeric(panel[column], errors="coerce")

# rows without iso code or year
before = len(panel)
panel = panel.dropna(subset=["iso3", "year"])
panel = panel[panel["iso3"].str.strip() != ""]
panel["year"] = panel["year"].astype(int)

# duplicates
n_duplicates = int(panel.duplicated(subset=["iso3", "year"]).sum())
panel = panel.drop_duplicates(subset=["iso3", "year"])

# add country names and regions, inner join also removes aggregates
countries_meta["income_group"] = countries_meta["income_group"].replace("Not classified", np.nan)
panel = panel.merge(countries_meta, on="iso3", how="inner")

print("rows before:", before, "after:", len(panel), "duplicates removed:", n_duplicates)
print("countries:", panel["iso3"].nunique())

In [ ]:
# missing values
cols = list(INDICATORS.values())
print("missing, % of rows (before):")
print(panel[cols].isna().mean().mul(100).round(1))

# fill only short gaps (max 2 years) inside a country's series, other gaps stay NaN
panel = panel.sort_values(["iso3", "year"]).reset_index(drop=True)
filled = 0
for column in ["gdp_per_capita", "life_expectancy"]:
    n_before = panel[column].isna().sum()
    panel[column] = panel.groupby("iso3")[column].transform(
        lambda s: s.interpolate(limit=2, limit_area="inside")
    )
    filled += int(n_before - panel[column].isna().sum())

print("\ncells filled by interpolation:", filled)
print(panel[cols].isna().mean().mul(100).round(1))

### 3.2 Wikipedia data

In [ ]:
wiki = wiki_raw.drop_duplicates(subset="wiki_country", keep="first").copy()

# development tier from HDI value (UNDP thresholds)
wiki["hdi_tier"] = pd.cut(wiki["hdi"], bins=[0, 0.55, 0.70, 0.80, 1.0],
                          labels=["Low", "Medium", "High", "Very high"], right=False)
print(len(wiki_raw), "->", len(wiki))
print(wiki["hdi_tier"].value_counts())

## 4. Merging
The two sources have no common id, only country names, and they are written differently ("Korea, Rep." and "South Korea"). So I make a normalised key from the name and use a dictionary for the known different spellings. After merging I print the countries that did not match.

In [ ]:
def normalize_name(name):
    name = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    name = name.lower().replace("&", "and").replace("'", "")
    name = re.sub(r"[^a-z0-9]+", " ", name).strip()
    return re.sub(r"^the ", "", name)


ALIASES = {
    "south korea": "korea rep", "korea south": "korea rep", "north korea": "korea dem peoples rep",
    "russia": "russian federation", "iran": "iran islamic rep", "egypt": "egypt arab rep",
    "turkey": "turkiye", "vietnam": "viet nam", "laos": "lao pdr", "syria": "syrian arab republic",
    "yemen": "yemen rep", "venezuela": "venezuela rb", "gambia": "gambia the", "bahamas": "bahamas the",
    "dr congo": "congo dem rep", "democratic republic of the congo": "congo dem rep",
    "republic of the congo": "congo rep", "congo": "congo rep",
    "kyrgyzstan": "kyrgyz republic", "slovakia": "slovak republic", "czech republic": "czechia",
    "brunei": "brunei darussalam", "cape verde": "cabo verde", "ivory coast": "cote divoire",
    "hong kong": "hong kong sar china", "macau": "macao sar china", "macao": "macao sar china",
    "palestine": "west bank and gaza", "state of palestine": "west bank and gaza",
    "micronesia": "micronesia fed sts", "federated states of micronesia": "micronesia fed sts",
    "saint lucia": "st lucia", "saint kitts and nevis": "st kitts and nevis",
    "saint vincent and the grenadines": "st vincent and the grenadines",
    "east timor": "timor leste", "swaziland": "eswatini", "burma": "myanmar", "macedonia": "north macedonia",
}

def make_key(name):
    key = normalize_name(name)
    return ALIASES.get(key, key)


snapshot = panel[panel["year"] == ANALYSIS_YEAR].copy()
snapshot["key"] = snapshot["country"].map(make_key)
wiki["key"] = wiki["wiki_country"].map(make_key)

# outer merge only to see what does not match
check = snapshot.merge(wiki, on="key", how="outer", indicator=True)
print(check["_merge"].value_counts())
print("\nin Wikipedia but not in API:")
print(sorted(check.loc[check["_merge"] == "right_only", "wiki_country"].tolist()))
print("\nin API but without HDI:", int((check["_merge"] == "left_only").sum()))

merged = snapshot.merge(wiki[["key", "wiki_country", "hdi", "hdi_tier"]], on="key", how="inner").drop(columns="key")
print("\nmerged rows:", len(merged))

In [ ]:
# log transform because GDP and population are very skewed
merged["log_gdp"] = np.log10(merged["gdp_per_capita"])
merged["log_pop"] = np.log10(merged["population"])
merged["pop_millions"] = merged["population"] / 1e6

income_order = ["Low income", "Lower middle income", "Upper middle income", "High income"]
merged["income_group"] = pd.Categorical(merged["income_group"], categories=income_order, ordered=True)

analysis = merged.dropna(subset=["gdp_per_capita", "life_expectancy", "hdi"]).reset_index(drop=True)
analysis["gdp_rank"] = analysis["gdp_per_capita"].rank(ascending=False).astype(int)
analysis["hdi_rank"] = analysis["hdi"].rank(ascending=False, method="min").astype(int)

print("countries in analysis:", len(analysis))
display(analysis[["country", "region", "income_group", "gdp_per_capita", "population",
                  "life_expectancy", "pop_growth", "unemployment", "hdi", "hdi_tier"]].head())

panel.to_csv(DATA_DIR / "worldbank_panel_clean.csv", index=False)
wiki.drop(columns="key").to_csv(DATA_DIR / "wikipedia_hdi_clean.csv", index=False)
analysis.to_csv(DATA_DIR / "analysis_dataset.csv", index=False)

## 5. Exploratory data analysis
### 5.1 Descriptive statistics

In [ ]:
num_cols = ["gdp_per_capita", "population", "life_expectancy", "pop_growth", "unemployment", "hdi"]
display(analysis[num_cols].describe().T)

skew_raw = analysis["gdp_per_capita"].skew()
skew_log = analysis["log_gdp"].skew()
print("skewness of GDP per capita: raw =", round(skew_raw, 2), ", log10 =", round(skew_log, 2))

### 5.2 Groups

In [ ]:
by_income = analysis.groupby("income_group", observed=True).agg(
    countries=("iso3", "count"),
    gdp_median=("gdp_per_capita", "median"),
    life_median=("life_expectancy", "median"),
    hdi_median=("hdi", "median"),
    pop_growth_median=("pop_growth", "median"),
)
display(by_income)

by_region = analysis.groupby("region").agg(
    countries=("iso3", "count"),
    gdp_median=("gdp_per_capita", "median"),
    life_median=("life_expectancy", "median"),
    hdi_median=("hdi", "median"),
).sort_values("hdi_median", ascending=False)
display(by_region)

print("Top 5 GDP per capita")
display(analysis.nlargest(5, "gdp_per_capita")[["country", "gdp_per_capita", "hdi"]])
print("Bottom 5 GDP per capita")
display(analysis.nsmallest(5, "gdp_per_capita")[["country", "gdp_per_capita", "hdi"]])

### 5.3 Correlations
Pearson shows linear relation, Spearman works on ranks and is less sensitive to outliers. Correlation is not causation, and HDI itself includes income, so part of the link is expected.

In [ ]:
corr_cols = ["log_gdp", "log_pop", "life_expectancy", "pop_growth", "unemployment", "hdi"]
pearson = analysis[corr_cols].corr(method="pearson")
spearman = analysis[corr_cols].corr(method="spearman")
display(spearman)

r_hdi, p_hdi = stats.pearsonr(analysis["log_gdp"], analysis["hdi"])
r_life, p_life = stats.pearsonr(analysis["log_gdp"], analysis["life_expectancy"])
reg = stats.linregress(analysis["log_gdp"], analysis["hdi"])

print(f"log GDP vs HDI: r = {r_hdi:.3f}, p = {p_hdi:.1e}, R2 = {reg.rvalue**2:.3f}")
print(f"log GDP vs life expectancy: r = {r_life:.3f}, p = {p_life:.1e}")
print(f"HDI = {reg.intercept:.3f} + {reg.slope:.3f} * log10(GDP per capita)")

### 5.4 Kazakhstan and residuals

In [ ]:
# residual = real HDI minus HDI predicted from income
analysis["hdi_expected"] = reg.intercept + reg.slope * analysis["log_gdp"]
analysis["hdi_residual"] = analysis["hdi"] - analysis["hdi_expected"]

kaz_rows = analysis[analysis["iso3"] == "KAZ"]
kaz = kaz_rows.iloc[0] if len(kaz_rows) else None
if kaz is not None:
    print(f"Kazakhstan: GDP per capita {kaz.gdp_per_capita:,.0f} (rank {kaz.gdp_rank} of {len(analysis)}), "
          f"HDI {kaz.hdi:.3f} (rank {kaz.hdi_rank}), life expectancy {kaz.life_expectancy:.1f}")
    print(f"expected HDI {kaz.hdi_expected:.3f}, residual {kaz.hdi_residual:+.3f}")
else:
    print("Kazakhstan is not in the analysis table, check the merge")

print("\nHighest HDI compared to income:")
display(analysis.nlargest(5, "hdi_residual")[["country", "gdp_per_capita", "hdi", "hdi_residual"]])
print("Lowest HDI compared to income:")
display(analysis.nsmallest(5, "hdi_residual")[["country", "gdp_per_capita", "hdi", "hdi_residual"]])

## 6. Visualisations

In [ ]:
def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / name, dpi=200, bbox_inches="tight")
    plt.show()

# Plot 1: GDP vs HDI
fig, ax = plt.subplots(figsize=(9, 6))
regions = sorted(analysis["region"].unique())
palette = dict(zip(regions, sns.color_palette("tab10", len(regions))))
for name, grp in analysis.groupby("region"):
    ax.scatter(grp["gdp_per_capita"], grp["hdi"], s=20 + 8 * np.sqrt(grp["pop_millions"]),
               color=palette[name], alpha=0.7, edgecolor="white", label=name)

xs = np.linspace(analysis["log_gdp"].min(), analysis["log_gdp"].max(), 100)
ax.plot(10 ** xs, reg.intercept + reg.slope * xs, "k--", label=f"linear fit on log GDP (R2 = {reg.rvalue**2:.2f})")
if kaz is not None:
    ax.annotate("Kazakhstan", (kaz.gdp_per_capita, kaz.hdi), xytext=(15, -35),
                textcoords="offset points", arrowprops=dict(arrowstyle="->"))
ax.set_xscale("log")
ax.set_xlabel("GDP per capita, current US$ (log scale)")
ax.set_ylabel("Human Development Index")
ax.set_title(f"GDP per capita vs HDI, {ANALYSIS_YEAR}")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, title="Region (size = population)")
save_fig(fig, "01_gdp_vs_hdi.png")

In [ ]:
# Plot 2: life expectancy by income group
fig, ax = plt.subplots(figsize=(8, 5))
plot_data = analysis.dropna(subset=["income_group"])
sns.boxplot(data=plot_data, x="income_group", y="life_expectancy", order=income_order,
            hue="income_group", hue_order=income_order, palette="Blues", legend=False, ax=ax, fliersize=0)
sns.stripplot(data=plot_data, x="income_group", y="life_expectancy", order=income_order,
              color="black", alpha=0.35, size=3, ax=ax)
ax.set_xlabel("Income group")
ax.set_ylabel("Life expectancy at birth, years")
ax.set_title(f"Life expectancy by income group, {ANALYSIS_YEAR}")
save_fig(fig, "02_life_expectancy_by_income.png")

In [ ]:
# Plot 3: correlation heatmap
labels = {"log_gdp": "log GDP", "log_pop": "log population", "life_expectancy": "Life expectancy",
          "pop_growth": "Population growth", "unemployment": "Unemployment", "hdi": "HDI"}
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(spearman.rename(index=labels, columns=labels), annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, square=True, ax=ax, cbar_kws={"label": "Spearman correlation"})
ax.set_title(f"Correlation matrix, {ANALYSIS_YEAR}")
save_fig(fig, "03_correlation_heatmap.png")

In [ ]:
# Plot 4: GDP per capita over time
compare = ["KAZ", "UZB", "RUS", "TUR", "CHN", "KOR"]
ts = panel[panel["iso3"].isin(compare)]
fig, ax = plt.subplots(figsize=(9, 5))
for iso, grp in ts.groupby("iso3"):
    ax.plot(grp["year"], grp["gdp_per_capita"], label=grp["country"].iloc[0],
            linewidth=3 if iso == "KAZ" else 1.5)
ax.set_yscale("log")
ax.set_xlabel("Year")
ax.set_ylabel("GDP per capita, current US$ (log scale)")
ax.set_title(f"GDP per capita {START_YEAR}-{END_YEAR}: Kazakhstan and other countries")
ax.legend(ncol=2)
save_fig(fig, "04_gdp_trend_kazakhstan.png")

In [ ]:
# Plot 5: raw vs log GDP
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(analysis["gdp_per_capita"], bins=30, ax=axes[0])
axes[0].set_title(f"GDP per capita (skew = {skew_raw:.2f})")
axes[0].set_xlabel("GDP per capita, US$")
axes[0].set_ylabel("Number of countries")
sns.histplot(analysis["log_gdp"], bins=30, ax=axes[1])
axes[1].set_title(f"log10 GDP per capita (skew = {skew_log:.2f})")
axes[1].set_xlabel("log10(GDP per capita)")
axes[1].set_ylabel("Number of countries")
save_fig(fig, "05_log_transform.png")

## 7. Key numbers for the report

In [ ]:
print("countries from API:", panel["iso3"].nunique(), "| panel rows:", len(panel))
print("scraped HDI rows:", len(wiki), "| matched:", len(merged), "| in analysis:", len(analysis))
print("duplicates removed:", n_duplicates, "| cells interpolated:", filled)
print("missing unemployment, %:", round(panel["unemployment"].isna().mean() * 100, 1))
print(f"skew GDP: {skew_raw:.2f} -> {skew_log:.2f} after log")
print(f"log GDP vs HDI: r = {r_hdi:.2f}, R2 = {reg.rvalue**2:.2f}; slope = {reg.slope:.2f} HDI per x10 GDP")
print(f"log GDP vs life expectancy: r = {r_life:.2f}")
print(f"log GDP vs population growth (Spearman): {spearman.loc['log_gdp', 'pop_growth']:.2f}")
print(f"log GDP vs unemployment (Spearman): {spearman.loc['log_gdp', 'unemployment']:.2f}")
display(by_income[["life_median", "pop_growth_median"]])